# 02. Training Loss Comparison Across All 4 Experimental Arms

This notebook reads the CSV loss logs generated by `train.py` and compares:
1. **Total loss** convergence
2. **PDE residual loss** convergence
3. **Boundary condition loss** convergence
4. **RAR data growth** (how many points/loads are added over training)

**Prerequisites**: Run `python kaggle_run_all.py` (or individual training scripts) first.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.abspath('../results')

arms = ['baseline', 'rar_collocation', 'rar_load', 'rar_combined']
colors = {
    'baseline': '#6C757D',
    'rar_collocation': '#0D6EFD',
    'rar_load': '#198754',
    'rar_combined': '#DC3545',
}
labels = {
    'baseline': 'Baseline (Uniform)',
    'rar_collocation': 'Collocation RAR',
    'rar_load': 'Load RAR',
    'rar_combined': 'Combined RAR',
}

# Load data
data = {}
for arm in arms:
    csv_path = os.path.join(RESULTS_DIR, f'{arm}_losses.csv')
    if os.path.exists(csv_path):
        data[arm] = pd.read_csv(csv_path)
        print(f'Loaded {arm}: {len(data[arm])} logged epochs')
    else:
        print(f'⚠ Not found: {csv_path}')

if not data:
    print('\nNo training data found. Run training first!')

## 1. Loss Convergence Comparison

In [ ]:
if data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    loss_cols = [
        ('total_loss', 'Total Loss'),
        ('pde_loss', 'PDE Residual Loss'),
        ('bc_loss', 'Boundary Condition Loss'),
    ]

    for ax, (col, title) in zip(axes, loss_cols):
        for arm, df in data.items():
            ax.semilogy(df['epoch'], df[col],
                        label=labels[arm], color=colors[arm],
                        alpha=0.85, linewidth=1.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.set_title(title)
        ax.legend(fontsize=9)
        ax.grid(True, which='both', alpha=0.3)

    plt.suptitle('PI-DeepONet-RAR: Training Loss Convergence', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Print final losses
    print('\nFinal Losses:')
    for arm, df in data.items():
        last = df.iloc[-1]
        print(f'  {labels[arm]:25s}: Total={last["total_loss"]:.4e}, '
              f'PDE={last["pde_loss"]:.4e}, BC={last["bc_loss"]:.4e}')

## 2. RAR Data Growth

Shows how collocation points and load functions increase during RAR refinement.

In [ ]:
if data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for arm, df in data.items():
        ax1.plot(df['epoch'], df['num_domain_pts'],
                 label=labels[arm], color=colors[arm], linewidth=1.5)
        ax2.plot(df['epoch'], df['num_loads'],
                 label=labels[arm], color=colors[arm], linewidth=1.5)

    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Domain Collocation Points')
    ax1.set_title('Spatial Point Growth')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Active Load Functions')
    ax2.set_title('Load Function Growth')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.suptitle('RAR: Adaptive Training Data Growth', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Convergence Rate Analysis

In [ ]:
if data:
    print('Convergence Summary:')
    print(f'{"Arm":25s} {"First Loss":>12s} {"Final Loss":>12s} {"Reduction":>12s} {"Time (s)":>10s}')
    print('-' * 75)
    for arm, df in data.items():
        first = df['total_loss'].iloc[0]
        final = df['total_loss'].iloc[-1]
        reduction = first / final if final > 0 else float('inf')
        elapsed = df['elapsed_sec'].iloc[-1] if 'elapsed_sec' in df.columns else 0
        print(f'{labels[arm]:25s} {first:12.4e} {final:12.4e} {reduction:12.1f}x {elapsed:10.1f}')